# Jump-Diffusion Models in Finance

From-scratch exploration of Poisson processes, Merton's jump-diffusion model, option pricing with jumps, and the implied volatility smile.

**Outline**
1. Why GBM isn't enough -- crashes happen!
2. The Poisson process
3. Compound Poisson process
4. Merton's jump-diffusion model
5. Return distribution -- mixture of normals
6. Merton's option pricing formula
7. Series convergence
8. The implied volatility smile
9. Monte Carlo pricing
10. Sensitivity to jump parameters
11. Calibration
12. Summary
13. References

### Why this notebook matters

Black-Scholes-Merton is wrong in one important way: real markets *jump*. Stock prices crash, news triggers gaps, central banks announce decisions, earnings surprise, M&A deals are revealed. None of these continuous-path moves exist in the BSM world.

Jump-diffusion models solve this. They retain the Brownian motion that captures normal trading-noise volatility, but *add* a jump process to capture the discrete shocks. The result is a model that:

* Generates **fat-tailed return distributions** that match empirical data
* Produces **volatility smiles** consistent with options markets
* Captures **crisis episodes** that BSM cannot represent
* Provides **closed-form** option pricing formulas (Merton 1976)

> **Key Concept:** Jump-diffusion is the simplest model that captures the *qualitative* behaviour of real financial markets. It adds a single new ingredient — a Poisson jump process — to GBM, and the result transforms model behaviour. Mastering jump-diffusion is the gateway to advanced derivatives pricing, exotic options, and risk management for tail events.

### Outline of this notebook

1. The Poisson process — counting random events in time
2. Compound Poisson — adding jump magnitudes
3. The Merton jump-diffusion model — combining diffusion and jumps
4. Comparing distributions: BSM vs jump-diffusion
5. Pricing options — the Merton series formula
6. Implementation and convergence
7. The volatility smile and how jump-diffusion produces it
8. Monte Carlo for path-dependent options
9. Visualisation: paths, distributions, smiles
10. Calibration to market data
11. References and further reading

Each section builds on the previous, progressing from the building blocks (Poisson processes) to the complete model (Merton's jump-diffusion) to its applications (option pricing, risk management).

> **Recommended approach:** Work through this notebook sequentially. The mathematics builds cumulatively — skipping ahead means missing crucial conceptual steps. The simulation code accompanies each theoretical concept, providing concrete examples that reinforce the abstract material.

---
## 1. Why GBM Isn't Enough -- Crashes Happen!

GBM assumes continuous price paths. Under GBM, a 20% single-day drop is essentially impossible.

But real markets disagree:
- **Black Monday (1987):** S&P 500 fell 20.5% in one day -- a 25-sigma event under GBM.
- **Flash Crash (2010):** Dow dropped 9% in minutes.
- **COVID Crash (2020):** Multiple days of 5-10% moves.

Real returns have **fat tails** -- far more extreme moves than normal predicts.

| Feature | GBM | Reality |
|---------|-----|---------|
| Price paths | Continuous | Can jump suddenly |
| Return distribution | Normal (thin tails) | Fat tails (leptokurtic) |
| Skewness | Zero | Negative (crashes > rallies) |
| Implied volatility | Flat across strikes | Smile/skew pattern |

### Merton's insight (1976)

Keep GBM for "normal" fluctuations, add occasional **jumps** for crashes/rallies. Jumps arrive randomly (like earthquakes) with random sizes.

> **Key Concept:** The jump-diffusion model separates two types of risk: (1) **diffusion risk** -- small, continuous noise, and (2) **jump risk** -- rare, sudden large moves. Together, they produce fat tails and the volatility smile.

### Real-world analogy

Stock prices are like sea level: **diffusion** = gentle waves (continuous, small), **jumps** = tsunamis (rare, sudden). You need separate mechanisms for each.

### The history and motivation

Robert Merton's 1976 paper *"Option pricing when underlying stock returns are discontinuous"* introduced jump-diffusion to finance. Merton's insight was simple but powerful: the BSM continuous-path assumption was *unrealistic* — every trader knew that prices jumped — and this contributed to the puzzling differences between BSM-implied prices and actual market prices for out-of-the-money options.

The 1987 stock market crash (Black Monday, October 19, 1987) drove this point home dramatically. The S&P 500 fell 22.6% in a single day — a return so extreme that under BSM-normal assumptions, it should occur less than once in the age of the universe. Real markets, however, produce such events occasionally. The "fat tails" of return distributions are not statistical artifacts — they reflect actual market dynamics.

Post-1987, the **volatility smile** appeared in equity options markets: out-of-the-money puts trade at *higher* implied volatilities than at-the-money options. BSM cannot explain this — it predicts a flat volatility surface. Jump-diffusion can: jumps disproportionately affect deep OTM options, raising their implied volatilities.

### Where jump-diffusion fits in the model hierarchy

| Model | Continuous Variance? | Jumps? | Used For |
|-------|---------------------|--------|----------|
| Geometric Brownian Motion (BSM) | Constant | No | Baseline pricing, simple cases |
| Local volatility (Dupire) | Function of $S, t$ | No | Fitting smile statically |
| Stochastic volatility (Heston) | Stochastic | No | Smile + dynamics |
| **Jump-diffusion (Merton)** | **Constant** | **Yes** | **Tail risk + smile** |
| Stochastic vol + jumps (Bates) | Stochastic | Yes | Most flexible (and complex) |

Jump-diffusion is one rung up the complexity ladder from BSM, providing tail-risk capture at the cost of additional parameters and computational complexity.

> **CFA Exam Tip:** While the CFA curriculum focuses primarily on BSM and binomial models, jump-diffusion concepts appear in advanced derivatives readings (Level 2 and 3). Understanding *why* BSM fails — and how jump-diffusion fixes it — is essential for grasping modern derivatives pricing and risk management.

### The "fat tail" problem in finance

Real financial returns have **fat tails** — extreme events occur far more frequently than normal-distribution-based models predict. The numerical evidence:

| Statistic | Normal Distribution | S&P 500 (1957-2024) |
|-----------|---------------------|---------------------|
| Skewness | 0 | -0.5 |
| Kurtosis | 3 | ~35 |
| Frequency of -5% daily returns | ~ 1 in 5,800 days | ~ 1 in 75 days |
| Frequency of -10% daily returns | ~ 1 in $10^{12}$ days | ~ 1 in 5,000 days |
| Worst single day in history | impossible under normal | -22.6% (1987) |

These differences are not statistical curiosities — they have profound implications:
* **Risk management:** Standard VaR models based on normality dramatically underestimate tail risk
* **Portfolio insurance:** "Probably won't fall below X" can fail catastrophically
* **Option pricing:** OTM options are far more valuable than BSM predicts
* **Regulatory capital:** Bank capital requirements assume distributions that don't reflect reality

Jump-diffusion models address all of these failures by explicitly modelling the discrete events that create fat tails.

### Notable jumps in market history

A non-exhaustive list of major historical jumps that demonstrate the empirical importance of jump-diffusion:

| Date | Event | Magnitude | Why It Mattered |
|------|-------|-----------|----------------|
| Oct 19, 1987 | Black Monday | -22.6% (S&P 500) | Largest single-day decline in modern history; impossible under BSM |
| Sep 11, 2001 | Terrorist attacks | -7.1% (S&P 500) | Markets closed for 4 days; massive jump on reopening |
| Sep 15, 2008 | Lehman Brothers bankruptcy | -4.7% then continuing | Triggered global financial crisis |
| Aug 24, 2015 | "Flash crash" | -3.9% intraday | Volatility ETN spike, ETF dislocations |
| Mar 16, 2020 | COVID-19 crash | -12.0% (S&P 500) | Pandemic uncertainty + circuit breaker triggered |
| Aug 5, 2024 | Carry trade unwind | -3.0% (S&P 500), -12% (Nikkei) | Yen carry trade liquidation |

Each of these events would have required dozens of standard deviations under BSM-normal assumptions — events that should occur less than once in millions of trading days. Yet they happened, demonstrating the failure of the normal-distribution assumption for tail events.

### Why this matters for finance professionals

Every CFA-certified investment professional must understand:
1. Real markets jump
2. BSM cannot capture jumps
3. Tail risk requires explicit modelling
4. Jump-diffusion is the simplest viable framework

This understanding informs portfolio construction (avoid over-leveraging), risk management (size positions for tail events), product design (price tail-risk insurance), and capital allocation (hold reserves for unexpected events).

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# ── Tolerances
ATOL = 1e-10
RTOL = 1e-6

# ── Plot Style
PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

---
## 2. The Poisson Process -- Random Arrivals

### Real-world examples
- Earthquakes (~1 major per decade)
- Customer arrivals (~5 per hour)
- Market crashes (~1 per 5-10 years)

### Definition

$$P(N(t) = k) = \frac{(\lambda t)^k e^{-\lambda t}}{k!}$$

**Variables:** $N(t)$ = number of events by time $t$, $\lambda$ = intensity (events per unit time).

**Key properties:** $E[N(t)] = \lambda t$, $\text{Var}[N(t)] = \lambda t$, inter-arrival times are Exp($\lambda$).

### Worked example

Crashes at rate $\lambda = 0.5$/year:
- P(0 crashes in 1 year) = $e^{-0.5} = 61\%$
- P(1 crash) = $0.5 e^{-0.5} = 30\%$
- P(2+ crashes) = $9\%$
- Expected wait: $1/\lambda = 2$ years

### The Poisson process — counting random arrivals

The Poisson process is the foundation of jump models. It counts the number of "events" that occur in time, where:

1. Events occur at random, unpredictable times
2. The expected number of events in interval $[t, t+\Delta]$ is $\lambda \Delta$
3. Events in disjoint intervals are independent

The parameter $\lambda$ (the **intensity** or **rate**) is the expected number of events per unit time. If $\lambda = 5$ per year, you expect 5 events per year on average — but the actual number varies.

### Properties of the Poisson process

The number of events $N_t$ in $[0, t]$ has a **Poisson distribution**:

$$P(N_t = k) = \frac{(\lambda t)^k e^{-\lambda t}}{k!}$$

with mean $E[N_t] = \lambda t$ and variance $\text{Var}(N_t) = \lambda t$.

The **inter-arrival times** $T_i$ — the gaps between consecutive events — are *exponentially distributed*:

$$T_i \sim \text{Exp}(\lambda), \quad P(T_i > s) = e^{-\lambda s}$$

This exponential property is the **memoryless** property: given that no event has occurred so far, the time until the next event has the same distribution as if you started counting from now.

### Why Poisson processes appear so often

Many real-world phenomena are well-approximated by Poisson processes:
* **Defaults** — corporate defaults arrive at random
* **Trades** — order arrivals on an exchange
* **News events** — announcements that affect a stock
* **Market crashes** — large adverse moves
* **Insurance claims** — accidents, hurricanes, deaths

The Poisson process is the *universal model* for "random arrivals of independent events at a constant rate."

> **Key Concept:** The Poisson distribution is the discrete analogue of the *exponential* distribution. The number of events in a fixed time → Poisson. The time between events → Exponential. They are inseparable, and any problem involving one almost always involves the other.

### Self-study exercises

To deepen understanding of Poisson processes:

1. **Simulation exercise:** Generate Poisson processes with $\lambda = 1, 5, 50$ events/year. Verify that:
   * The mean number of events approaches $\lambda T$
   * The variance of the count also approaches $\lambda T$
   * Inter-arrival times have mean $1/\lambda$

2. **Distribution exercise:** Plot the Poisson PMF for $\lambda T = 1, 5, 10$ and observe how the shape transitions from highly skewed (low $\lambda T$) to nearly normal (high $\lambda T$) — illustrating the Central Limit Theorem.

3. **Connection to exponential:** Verify that the time of the $n$-th event has a Gamma distribution with shape $n$ and scale $1/\lambda$.

These exercises build intuition that pays dividends in the more complex jump-diffusion settings later.

### Setup notes

The simulations in this notebook use synthetic but realistic parameters:
* Equity-like volatility: $\sigma = 20-25\%$
* Jump frequency: $\lambda = 1-5$ per year (typical for equity indices)
* Average jump size: $\mu_J = -5\%$ to $-10\%$ (downward bias, reflecting crash risk)
* Jump dispersion: $\sigma_J = 5-10\%$

These parameters produce paths that resemble actual equity index dynamics. Calibrating to specific markets (single stocks, indices, FX, commodities) produces different parameter values, but the qualitative behaviour is similar.

In [ ]:
def simulate_poisson_process(lam, T, rng):
    """Simulate a Poisson process via inter-arrival times.
    
    Args:
        lam: Intensity (expected jumps per unit time)
        T: Time horizon
        rng: NumPy random generator
    
    Returns:
        jump_times: array of jump times in [0, T]
    """
    jump_times = []
    t = 0.0
    while True:
        # Inter-arrival time ~ Exp(lambda)
        dt = rng.exponential(1.0 / lam)
        t += dt
        if t > T:
            break
        jump_times.append(t)
    return np.array(jump_times)

# Simulate and visualize
lam = 3.0  # 3 jumps per year on average
T = 5.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: sample paths
for i in range(5):
    jt = simulate_poisson_process(lam, T, rng)
    times = np.concatenate([[0], np.repeat(jt, 2), [T]])
    counts = np.repeat(np.arange(len(jt) + 1), 2)
    axes[0].plot(times, counts, alpha=0.7, label=f'Path {i+1}')

axes[0].set_xlabel('Time')
axes[0].set_ylabel('N(t)')
axes[0].set_title(f'Poisson Process Sample Paths ($\\lambda = {lam}$)')
axes[0].legend(fontsize=9)

# Right: distribution of N(T) vs theoretical
n_sims = 10000
N_T = rng.poisson(lam * T, size=n_sims)
k_vals = np.arange(0, max(N_T) + 1)
pmf_theory = stats.poisson.pmf(k_vals, lam * T)

axes[1].hist(N_T, bins=k_vals - 0.5, density=True, alpha=0.6, color=PRIMARY, label='Simulated')
axes[1].plot(k_vals, pmf_theory, 'o-', color=SECONDARY, label='Theoretical PMF')
axes[1].set_xlabel('k')
axes[1].set_ylabel('P(N(T) = k)')
axes[1].set_title(f'Distribution of N({T}) with $\\lambda = {lam}$')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 3. Compound Poisson Process -- Random-Sized Jumps

$$X(t) = \sum_{i=1}^{N(t)} J_i$$

In Merton's model: $\ln(1 + J_i) \sim \mathcal{N}(\mu_J, \sigma_J^2)$ (lognormal jump factors ensure positive prices).

### Worked example

$\mu_J = -0.05$, $\sigma_J = 0.10$: average factor $e^{-0.045} = 0.956$ (about -4.4%). A 2-sigma event: stock drops 22% or rises 16%.

### The compound Poisson process

A **compound Poisson process** combines two ingredients:
1. A **Poisson process** governing *when* events occur
2. A **distribution of jump sizes** governing *how big* each event is

The total cumulative impact at time $t$ is:

$$Y_t = \sum_{i=1}^{N_t} J_i$$

where $J_i$ are independent and identically distributed jump sizes. The randomness comes from both the *number* of jumps ($N_t$) and the *magnitudes* ($J_i$).

### Common jump size distributions

In financial applications, common choices for $J_i$:

| Distribution | Use Case | Properties |
|--------------|----------|-----------|
| **Normal** | Symmetric jumps | Tractable, allows positive and negative |
| **Lognormal** | Multiplicative jumps | Used in Merton's model for stock prices |
| **Double exponential** | Asymmetric jumps | Heavier tails, used in Kou model |
| **Constant** | Fixed jump magnitude | Simplest case |

Merton's classic model uses lognormal jump sizes — this gives the elegant property that the underlying stock price is multiplied by a lognormal factor at each jump, preserving positivity.

### Mean and variance

For a compound Poisson process:

$$E[Y_t] = \lambda t \cdot E[J]$$
$$\text{Var}(Y_t) = \lambda t \cdot E[J^2]$$

The variance scales linearly with time, just like Brownian motion. This is why jump processes can be combined cleanly with diffusion processes.

> **Common Mistake:** A subtle but important point: $\text{Var}(Y_t) = \lambda t \cdot E[J^2]$, NOT $\lambda t \cdot \text{Var}(J)$. The variance includes the *second moment* of jump sizes, which equals $\text{Var}(J) + E[J]^2$. So large *mean* jumps contribute to variance even if their dispersion is small.

### Visualisation: the compound Poisson process

When plotted, a compound Poisson process looks like:
* A flat line during periods between events
* Sudden discrete jumps at random times
* Jump sizes drawn from the chosen distribution
* Cumulative process steps in random magnitudes

Compare this to a Brownian motion path, which is continuously fluctuating with no flat periods. The contrast is exactly what the jump component adds to a jump-diffusion model.

### Time-changed Brownian motion — an alternative view

The compound Poisson process can be viewed as a *random time change* of an underlying process. This perspective unifies many jump models:

* **Variance Gamma:** Brownian motion time-changed by a Gamma process
* **Normal Inverse Gaussian (NIG):** Brownian motion time-changed by an inverse Gaussian process
* **CGMY:** Generalised time change with adjustable parameters

The "time change" interpretation says: instead of running clock time, the asset price runs on a *trading time* clock that ticks irregularly. Periods of high information arrival (busy markets) tick fast; quiet periods tick slow. This produces clustering of activity and fat-tailed returns naturally.

### Connection to the broader Lévy process family

A **Lévy process** is a stochastic process with independent and stationary increments. Both Brownian motion and the compound Poisson process are Lévy processes. So is their sum (the jump-diffusion process).

The Lévy-Khintchine theorem says: any Lévy process can be decomposed into:
1. A drift component
2. A continuous Brownian component
3. A jump component (which itself is a Poisson-like process integrating over jump sizes)

This decomposition is the *most general* framework for processes used in finance. Jump-diffusion is one specific instance; Variance Gamma is another; CGMY is another. All are Lévy processes.

> **Key Concept:** Lévy processes generalise both Brownian motion and Poisson processes. The class is rich enough to capture virtually all empirically observed return dynamics, while remaining tractable enough for derivatives pricing. Modern advanced derivatives theory is largely the theory of Lévy processes.

In [ ]:
def simulate_compound_poisson(lam, mu_J, sigma_J, T, rng):
    """Simulate a compound Poisson process with lognormal jumps.
    
    Args:
        lam: Jump intensity
        mu_J: Mean of log-jump size
        sigma_J: Std of log-jump size
        T: Time horizon
        rng: NumPy random generator
    
    Returns:
        jump_times: array of jump times
        jump_sizes: array of multiplicative jump factors (1 + J_i)
    """
    jump_times = simulate_poisson_process(lam, T, rng)
    n_jumps = len(jump_times)
    if n_jumps == 0:
        return jump_times, np.array([])
    # Log-jump sizes ~ N(mu_J, sigma_J^2)
    log_jumps = rng.normal(mu_J, sigma_J, size=n_jumps)
    jump_factors = np.exp(log_jumps)  # multiplicative: 1+J = exp(log_jump)
    return jump_times, jump_factors

# Example: compound Poisson with negative mean jumps (crash-like)
lam_ex = 2.0
mu_J_ex = -0.05   # mean log-jump: slight negative (crashes)
sigma_J_ex = 0.10  # volatility of jump size

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(5):
    jt, jf = simulate_compound_poisson(lam_ex, mu_J_ex, sigma_J_ex, 3.0, rng)
    # Build cumulative product path
    if len(jt) > 0:
        cum_jump = np.concatenate([[1.0], np.cumprod(jf)])
        times = np.concatenate([[0], jt])
    else:
        cum_jump = np.array([1.0])
        times = np.array([0])
    axes[0].step(times, cum_jump, where='post', alpha=0.7)

axes[0].set_xlabel('Time')
axes[0].set_ylabel('Cumulative Jump Factor')
axes[0].set_title('Compound Poisson Process (Multiplicative)')
axes[0].axhline(y=1, color='black', linestyle='--', alpha=0.5)

# Distribution of jump sizes
jump_samples = np.exp(rng.normal(mu_J_ex, sigma_J_ex, 10000)) - 1  # J_i values
axes[1].hist(jump_samples, bins=80, density=True, alpha=0.6, color=PRIMARY)
axes[1].axvline(x=0, color='black', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Jump Size $J_i$')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Jump Size Distribution ($\\mu_J = {mu_J_ex}$, $\\sigma_J = {sigma_J_ex}$)')

plt.tight_layout()
plt.show()

---
## 4. Merton's Jump-Diffusion Model

$$\frac{dS}{S} = (\mu - \lambda k) \, dt + \sigma \, dW + J \, dN$$

| Term | Meaning | Analogy |
|------|---------|---------|
| $(\mu - \lambda k) \, dt$ | Drift (adjusted) | River current |
| $\sigma \, dW$ | Diffusion | Gentle waves |
| $J \, dN$ | Jumps | Tsunamis |

**Variables:** $k = E[J] = e^{\mu_J + \sigma_J^2/2} - 1$. The $-\lambda k$ term **compensates** for the average jump, ensuring overall expected return is exactly $\mu$.

> **Key Concept:** The compensator separates "risk" of jumps from "reward." For risk-neutral pricing, expected return must be $r$ regardless of jump parameters.

### The Merton jump-diffusion model

Combining Brownian motion with jumps, Merton's stock price model is:

$$\frac{dS_t}{S_{t-}} = (\mu - \lambda \kappa) dt + \sigma dW_t + (J - 1) dN_t$$

where:
* $\mu$ is the drift
* $\sigma$ is the diffusion volatility
* $W_t$ is a Brownian motion
* $N_t$ is a Poisson process with intensity $\lambda$
* $J$ is the (random) jump multiplier — when a jump occurs, $S$ jumps from $S_{t-}$ to $S_{t-} \cdot J$
* $\kappa = E[J] - 1$ is the expected jump return
* The $\lambda \kappa$ adjustment ensures $E[dS_t] = \mu S_t \, dt$ (preserving the original drift after accounting for jumps)

### What the model captures

The Merton model produces a stock price that:
1. **Drifts upward** at rate $\mu$ (when $\mu > 0$)
2. **Diffuses** with volatility $\sigma$ between jumps
3. **Jumps** at random Poisson times with magnitude $J$
4. **Stays positive** if $J > 0$ always (lognormal $J$ ensures this)

Each "path" of the simulated stock price thus shows:
* Periods of normal trading (smooth diffusion)
* Punctuated by sudden discontinuous moves (jumps)

The visual pattern matches what real stock prices look like during news events, earnings releases, or crisis episodes.

### Why lognormal jumps?

Merton chose $\log J \sim N(\mu_J, \sigma_J^2)$. This has several advantages:
* $J > 0$ always → stock prices stay positive
* $\log S$ is a sum of normal increments + Poisson sum of normal jumps (analytically tractable)
* Closed-form option pricing formula (Merton 1976) becomes possible

The distribution of $\log J$ has parameters:
* **$\mu_J$**: average jump size (in log terms)
* **$\sigma_J$**: jump size dispersion

Negative $\mu_J$ produces *crash* models (jumps are typically downward); positive $\mu_J$ would model upward news shocks.

> **Key Concept:** The Merton model has *five* parameters ($\mu, \sigma, \lambda, \mu_J, \sigma_J$) compared to BSM's *three* ($\mu, \sigma$, and risk-free $r$). The two extra parameters describe the jump component. This added flexibility comes at the cost of harder calibration — five parameters can fit a wider variety of patterns but require more data to estimate reliably.

### Why we need the drift adjustment

The drift adjustment $-\lambda \kappa$ in the Merton equation is critical. Without it, jumps would systematically alter the *expected return* of the asset.

Consider: if $E[J] = 1.05$ (jumps add 5% on average), then with $\lambda = 1$ jump per year:
* Without adjustment: expected annual return = $\mu + 5\% = 8 + 5 = 13\%$
* With $\lambda \kappa$ adjustment: expected annual return = $\mu = 8\%$ (matches BSM)

The adjustment is what makes Merton's model "comparable" to BSM in terms of expected returns. Both models drift at $\mu$; they differ only in the *path* taken to get there.

### Risk-neutral version

For option pricing, we use the risk-neutral version of the Merton model:

$$\frac{dS_t}{S_{t-}} = (r - \lambda \kappa) dt + \sigma dW_t^Q + (J - 1) dN_t^Q$$

where $r$ replaces $\mu$ as the drift. Under the risk-neutral measure $Q$:
* Drift = risk-free rate (minus jump compensator)
* Diffusion volatility = same as physical
* Jump intensity may differ from physical (due to risk premium for jump risk)
* Jump size distribution may differ similarly

This complication — jump risk has a price not captured by simple replication — is why jump-diffusion isn't *complete* in the technical financial sense. Hedging strategies cannot fully eliminate risk; some residual exposure remains.

### Solving the Merton SDE

Despite its complexity, the Merton SDE has an explicit solution. Define $W_t$ as the standard Brownian motion and $N_t$ as the Poisson counting process. The solution for the stock price is:

$$S_t = S_0 \exp\left[(\mu - \frac{1}{2}\sigma^2 - \lambda \kappa) t + \sigma W_t\right] \prod_{i=1}^{N_t} J_i$$

The product over jumps captures the multiplicative effect: each jump multiplies the stock price by the random factor $J_i$. Between jumps, the price follows GBM.

### Taking logs

Taking the logarithm:

$$\log S_t = \log S_0 + (\mu - \frac{1}{2}\sigma^2 - \lambda \kappa) t + \sigma W_t + \sum_{i=1}^{N_t} \log J_i$$

The log-price is a Brownian motion plus a compound Poisson process (since $\log J_i$ are i.i.d. random variables). This decomposition is what makes Merton's option pricing formula work — the log-price has a tractable structure.

### Distribution of $\log S_t$

Conditional on $N_t = n$:
$$\log S_t | N_t = n \sim N\left(\log S_0 + \mu_n t, \sigma_n^2 t\right)$$

where $\mu_n$ and $\sigma_n$ are adjusted for the $n$ jumps. Marginalising over the Poisson distribution of $N_t$ gives a Poisson-weighted mixture of normal distributions — explaining the heavier tails.

> **Common Mistake:** Beginners sometimes treat $\lambda$ in the Merton model as the actual jump frequency observed in historical data. It is not — it is the *risk-neutral* implied frequency calibrated from option prices. The actual physical jump frequency may differ; the gap is the jump risk premium. Mixing these two gives wrong answers.

In [ ]:
def simulate_merton_jump_diffusion(S0, mu, sigma, lam, mu_J, sigma_J, T, n_steps, n_paths, rng):
    """Simulate Merton jump-diffusion paths.
    
    Uses exact simulation: for each time step, add the diffusion component
    and multiply by any jumps that occur in that interval.
    
    Args:
        S0: Initial stock price
        mu: Drift
        sigma: Diffusion volatility
        lam: Jump intensity
        mu_J: Mean of log-jump size
        sigma_J: Std of log-jump size
        T: Time horizon
        n_steps: Number of time steps
        n_paths: Number of simulation paths
        rng: NumPy random generator
    
    Returns:
        t: time grid (n_steps + 1,)
        S: price paths (n_paths, n_steps + 1)
    """
    dt = T / n_steps
    k = np.exp(mu_J + 0.5 * sigma_J**2) - 1  # E[J]
    
    t = np.linspace(0, T, n_steps + 1)
    S = np.zeros((n_paths, n_steps + 1))
    S[:, 0] = S0
    
    for i in range(n_steps):
        # Diffusion component
        Z = rng.standard_normal(n_paths)
        diffusion = (mu - lam * k - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
        
        # Jump component: number of jumps in [t_i, t_{i+1}]
        n_jumps = rng.poisson(lam * dt, size=n_paths)
        
        # Sum of log-jump sizes for each path
        jump_component = np.zeros(n_paths)
        for j in range(n_paths):
            if n_jumps[j] > 0:
                log_jumps = rng.normal(mu_J, sigma_J, size=n_jumps[j])
                jump_component[j] = np.sum(log_jumps)
        
        S[:, i+1] = S[:, i] * np.exp(diffusion + jump_component)
    
    return t, S

# Parameters
S0 = 100
mu = 0.08
sigma = 0.20
lam = 1.0        # 1 jump per year on average
mu_J = -0.10     # mean log-jump: negative (crashes)
sigma_J = 0.15   # jump size volatility
T = 2.0
n_steps = 500
n_paths = 10

t, S = simulate_merton_jump_diffusion(S0, mu, sigma, lam, mu_J, sigma_J, T, n_steps, n_paths, rng)

# Compare with pure GBM paths
rng_gbm = np.random.default_rng(SEED + 1)
dt_gbm = T / n_steps
S_gbm = np.zeros((n_paths, n_steps + 1))
S_gbm[:, 0] = S0
for i in range(n_steps):
    Z = rng_gbm.standard_normal(n_paths)
    S_gbm[:, i+1] = S_gbm[:, i] * np.exp((mu - 0.5*sigma**2)*dt_gbm + sigma*np.sqrt(dt_gbm)*Z)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for p in range(n_paths):
    axes[0].plot(t, S_gbm[p], alpha=0.5, color=PRIMARY)
axes[0].set_title('GBM Paths (No Jumps)')
axes[0].set_xlabel('Time (years)')
axes[0].set_ylabel('Stock Price')

for p in range(n_paths):
    axes[1].plot(t, S[p], alpha=0.5, color=SECONDARY)
axes[1].set_title('Merton Jump-Diffusion Paths')
axes[1].set_xlabel('Time (years)')
axes[1].set_ylabel('Stock Price')

plt.tight_layout()
plt.show()

---
## 5. Return Distribution: The Mixture of Normals

Conditional on $n$ jumps:

$$\ln(S_T/S_0) | N(T)=n \sim \mathcal{N}((\mu - \lambda k - \sigma^2/2)T + n\mu_J, \sigma^2 T + n\sigma_J^2)$$

The unconditional distribution is a **Poisson mixture of normals**, producing:
- **Heavier tails** (excess kurtosis)
- **Negative skewness** (when $\mu_J < 0$)

> **Key Concept:** Mixing distributions with different variances always produces fatter tails than any single component. This is why jump-diffusion returns have excess kurtosis.

### Comparing distributions: BSM vs jump-diffusion

The simulated terminal price distributions reveal the key difference between BSM and jump-diffusion:

**BSM (lognormal):**
* Smooth, single-peaked distribution
* Moderate tails — extreme events very rare
* Symmetry around the median (in log space)
* Cannot produce "crashes" in any meaningful sense

**Merton jump-diffusion:**
* Heavier tails — extreme events more common
* Asymmetric (when $\mu_J < 0$) — left tail fatter than right
* Visible "shoulder" or secondary peak from rare large jumps
* Produces crash-like terminal values

### The empirical fit

How does jump-diffusion compare to *actual* stock return data?

**S&P 500 daily returns (1957-2024):**
* Normal distribution would predict skewness = 0, kurtosis = 3
* Actual: skewness ≈ -0.5, kurtosis ≈ 35

The skewness of -0.5 indicates a slightly fatter left tail (crashes are larger than rallies). The kurtosis of 35 vs predicted 3 is *enormous* — the actual distribution has dramatically more extreme outcomes than the normal predicts.

Jump-diffusion can produce these levels of skewness and kurtosis with appropriate parameters:
* Negative $\mu_J$ produces skewness
* High $\sigma_J$ produces kurtosis
* Low $\lambda$ produces "rare large jumps" rather than "frequent small jumps"

### Calibrating to fit data

The classical calibration problem: choose $(\sigma, \lambda, \mu_J, \sigma_J)$ to match observed:
1. Volatility (annualised standard deviation of returns)
2. Skewness
3. Kurtosis
4. Empirical CDF (or option-implied volatility surface)

This is over-determined (4 parameters vs many constraints), so calibration uses optimisation to minimise some loss function. Different practitioners use different loss functions, leading to slightly different calibrated parameters.

> **CFA Exam Tip:** When the exam discusses the fat tails of asset returns, jump-diffusion is one of the standard responses. Be prepared to explain *why* jumps create fat tails and *how* this affects option pricing, especially for OTM options where the jump component dominates.

### Practical implications for risk management

The empirical superiority of jump-diffusion over BSM has direct risk management implications:

**Value at Risk (VaR) underestimation:**
A 95% VaR computed using BSM-normal assumptions systematically underestimates true tail risk. The actual probability of breaching the VaR threshold is typically 8-15% rather than the assumed 5%. This is why regulators (Basel III) require backtesting of VaR models — to detect when assumptions break down.

**Stress testing:**
Stress scenarios should include not just "extreme but plausible" market moves but also "tail events" — magnitudes that would be impossible under normal assumptions but occur in real markets. Jump-diffusion provides a principled framework for generating such scenarios.

**Capital adequacy:**
Banks and insurers hold capital reserves to absorb losses. If they assume normality, their capital is calibrated against fat-tailed reality, often inadequately. The 2008 financial crisis exposed many institutions whose risk models under-counted tail risk.

**Hedging strategies:**
Delta hedging in BSM theoretically eliminates risk in continuous trading. With jumps, even continuous hedging cannot eliminate jump risk — the underlying can move discontinuously, leaving the hedger with unexpected losses. Practitioners therefore use "static hedges" with options to insure against jump risk.

> **Common Mistake:** A common misuse of normal-distribution-based models: applying them confidently to long horizons where extreme events become almost certain. A 100-year stress test under normal assumptions often produces "1-in-100-year" events that have actually occurred multiple times in the past century. Normal-based long-horizon analysis routinely misses obvious tail risks.

### The jump-diffusion alternative for risk

Replacing normal with jump-diffusion in risk models:
* **VaR:** Computed via Monte Carlo with jump-diffusion paths
* **Expected Shortfall:** Conditional on tail events, jump-diffusion gives realistic loss magnitudes
* **Stress scenarios:** Jump events become natural "what if" scenarios
* **Capital allocation:** Tail risk capital can be sized appropriately

This is why most major financial institutions now use jump-diffusion (or stochastic-volatility-with-jumps) as their default for tail risk modelling.

### Empirical jump detection

Given historical price data, can we identify *when jumps actually occurred*? Yes — using statistical methods:

**Rolling Z-score:** A return is flagged as a jump if its standardised value exceeds a threshold (e.g., 3 or 4 standard deviations).

**Bipower variation:** A non-parametric estimator that distinguishes diffusive volatility from jump volatility:
$$\text{BV}_t = \frac{\pi}{2} \sum_{i=2}^n |r_{i-1}| |r_i|$$

The realised variance $\text{RV}_t = \sum_i r_i^2$ exceeds bipower variation when jumps are present. The difference is approximately the jump contribution.

**Lee-Mykland test:** A formal hypothesis test for jumps in high-frequency data. Detects jumps with high statistical confidence.

These methods enable *jump filtering* — separating diffusive volatility from jump volatility in real-world data.

### Applications of jump filtering

* **Pure diffusion volatility estimation:** Removes jump contamination from $\sigma$ estimates
* **Jump frequency estimation:** Counts detected jumps to estimate $\lambda$
* **Jump size distribution:** Histograms of detected jumps to estimate $\mu_J, \sigma_J$
* **Crisis detection:** Real-time monitoring for jumps as crisis indicators

> **CFA Exam Tip:** Jump filtering is more advanced than typical CFA exam topics, but understanding that jumps can be detected empirically (not just modelled theoretically) provides important context. The data confirms jump-diffusion isn't just a theoretical convenience — actual jumps are statistically detectable in real markets.

In [ ]:
# Large-scale simulation for distribution comparison
n_sims = 100000
T_dist = 1.0 / 12  # monthly returns

# Jump-diffusion terminal values
_, S_jd = simulate_merton_jump_diffusion(S0, mu, sigma, lam, mu_J, sigma_J, T_dist, 1, n_sims, rng)
log_returns_jd = np.log(S_jd[:, -1] / S0)

# GBM terminal values
Z_gbm = rng.standard_normal(n_sims)
log_returns_gbm = (mu - 0.5 * sigma**2) * T_dist + sigma * np.sqrt(T_dist) * Z_gbm

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograms
bins = np.linspace(-0.30, 0.20, 100)
axes[0].hist(log_returns_gbm, bins=bins, density=True, alpha=0.5, color=PRIMARY, label='GBM')
axes[0].hist(log_returns_jd, bins=bins, density=True, alpha=0.5, color=SECONDARY, label='Jump-Diffusion')
axes[0].set_xlabel('Monthly Log Return')
axes[0].set_ylabel('Density')
axes[0].set_title('Return Distribution Comparison')
axes[0].legend()

# QQ plot
sorted_jd = np.sort(log_returns_jd)
theoretical_quantiles = stats.norm.ppf(np.linspace(0.001, 0.999, len(sorted_jd)))
# Standardize
jd_std = (sorted_jd - np.mean(sorted_jd)) / np.std(sorted_jd)
subsample = np.linspace(0, len(jd_std)-1, 1000, dtype=int)
axes[1].scatter(theoretical_quantiles[subsample], jd_std[subsample], s=2, alpha=0.5, color=SECONDARY)
axes[1].plot([-4, 4], [-4, 4], 'k--', alpha=0.5, label='Normal reference')
axes[1].set_xlabel('Theoretical Quantiles (Normal)')
axes[1].set_ylabel('Sample Quantiles (Jump-Diffusion)')
axes[1].set_title('QQ Plot: Jump-Diffusion vs Normal')
axes[1].legend()

plt.tight_layout()
plt.show()

# Summary statistics
print(f"{'Statistic':<20} {'GBM':>12} {'Jump-Diffusion':>15}")
print('-' * 50)
for name, data_gbm, data_jd in [
    ('Mean', np.mean(log_returns_gbm), np.mean(log_returns_jd)),
    ('Std Dev', np.std(log_returns_gbm), np.std(log_returns_jd)),
    ('Skewness', stats.skew(log_returns_gbm), stats.skew(log_returns_jd)),
    ('Kurtosis', stats.kurtosis(log_returns_gbm), stats.kurtosis(log_returns_jd)),
]:
    print(f"{name:<20} {data_gbm:>12.4f} {data_jd:>15.4f}")

---
## 6. Merton's Option Pricing Formula

$$C = \sum_{n=0}^{\infty} \frac{e^{-\lambda' T} (\lambda' T)^n}{n!} C_{\text{BS}}(S_0, K, T, r_n, \sigma_n)$$

**Reading this:** Average over all possible numbers of jumps. Each scenario is BS-priced with:
- $\sigma_n = \sqrt{\sigma^2 + n\sigma_J^2/T}$ (jumps add variance)
- $r_n = r - \lambda k + n\ln(1+k)/T$ (rate adjustment)
- $\lambda' = \lambda(1+k)$ (risk-neutral intensity)

> **Key Concept:** Since jumps increase variance, every additional jump raises the BS price. Jump-diffusion options are generally more expensive than BS -- the market is pricing crash risk.

### Pricing options under jump-diffusion: the Merton series

The most beautiful result in jump-diffusion: a closed-form option price exists.

Conditional on $n$ jumps occurring during $[0, T]$, the stock price dynamics are *equivalent* to a Black-Scholes process with:
* Modified volatility: $\sigma_n = \sqrt{\sigma^2 + n\sigma_J^2/T}$
* Modified rate: $r_n = r - \lambda\kappa + n(\mu_J + \sigma_J^2/2)/T$

This means the Merton option price is a *Poisson-weighted sum* of BSM prices:

$$C^{Merton} = \sum_{n=0}^{\infty} P(N_T = n) \cdot C^{BSM}(\sigma_n, r_n)$$

$$= \sum_{n=0}^{\infty} \frac{e^{-\lambda T}(\lambda T)^n}{n!} \cdot C^{BSM}(\sigma_n, r_n)$$

This is one of the most elegant results in derivatives pricing — a complex jump-diffusion model reduces to a *weighted sum of BSM prices* over different scenarios.

### Practical computation

The infinite series converges quickly because Poisson probabilities decay rapidly for $n$ much larger than $\lambda T$. In practice:
* Truncate at $n_{\max}$ where $P(N_T = n_{\max})$ becomes negligible (typically $n_{\max} \approx \lambda T + 5\sqrt{\lambda T}$)
* For $\lambda T = 1$ (one jump per year, one-year option), $n = 0$ to $n = 7$ usually suffices
* For larger $\lambda T$, the series converges more slowly

### Why the series works

The intuition: at expiry, the stock has experienced exactly $n$ jumps for some integer $n \geq 0$. Conditional on $n$, the dynamics are diffusion-only with adjusted parameters. Averaging over all possible $n$, weighted by Poisson probabilities, gives the unconditional expected payoff.

This conditioning argument is a textbook example of the **tower property** of conditional expectation in probability theory.

> **Key Concept:** The Merton series formula is the *only widely-used* closed-form formula for jump-diffusion options. Other models (stochastic volatility, double-jump) require numerical integration or Monte Carlo. The closed-form is what makes Merton's model practical for real-time risk management — option prices and Greeks can be computed in microseconds.

### A worked example: pricing an OTM put

Consider a stock at \$100 with one-year maturity, strike \$80 (20% OTM put). With BSM parameters $\sigma = 20\%$, $r = 5\%$:

**BSM put price:** approximately \$0.92

Under jump-diffusion with $\lambda = 1$ jump/year, $\mu_J = -10\%$ (10% downward jump on average), $\sigma_J = 5\%$ (small dispersion):

**Merton put price:** approximately \$2.40

The jump-diffusion price is **~2.6× higher** than BSM. Why? Because the jumps create scenarios where the stock can drop 10-20% in an instant, pushing the option deep into the money. BSM cannot generate such moves without extremely high overall volatility.

### Implied volatility translation

The Merton put price of \$2.40, fed back into the BSM formula as a market price, would imply a BSM volatility of approximately 28% — significantly higher than the diffusion volatility of 20%. This 8 percentage point difference is the **jump premium** — what BSM has to "borrow" from volatility to match the jump-diffusion price.

This is why deep OTM puts on equity indices trade at IVs of 30%+ while ATM IVs are 15-20% — the smile/smirk reflects market participants pricing in jump risk.

### The role of risk-neutral measure

Critical subtlety: option pricing requires the *risk-neutral* version of jump-diffusion, not the physical version.

Under the **physical measure $P$**:
* $\mu^P$ is the expected return (includes risk premium)
* $\lambda^P$ is the actual jump frequency
* Jump distribution under $P$ matches empirical observations

Under the **risk-neutral measure $Q$**:
* Drift = $r$ (risk-free rate, after jump compensation)
* $\lambda^Q$ may differ from $\lambda^P$ (incorporates jump risk premium)
* Jump distribution under $Q$ may differ from $P$

For option pricing, we use $Q$-measure parameters. For real-world risk management, we use $P$-measure parameters. Both are needed for different purposes.

### Why $Q \neq P$ for jumps

In a complete market (BSM), $Q$ and $P$ differ only in the drift — perfect hedging means there's no other risk premium. Jump-diffusion is *incomplete*: jump risk cannot be fully hedged away, so the market commands a risk premium for bearing it.

This premium manifests as:
* $\lambda^Q > \lambda^P$ (markets price in more frequent jumps than actually occur)
* OR $\mu_J^Q < \mu_J^P$ (markets price in larger downward jumps)

Either or both adjustments are possible. The total effect: option prices reflect a "risk-neutral" view of jumps that is *more pessimistic* than the actual jump distribution.

### Calibration in practice

When fitting Merton's model to option prices, the calibrated parameters are *risk-neutral*. They reflect:
* Actual jump dynamics
* Plus risk premia for jump risk

This is why calibrated $\lambda$ is often higher than empirical jump frequencies estimated from price returns. The CFA curriculum tests this distinction at higher levels.

> **Common Mistake:** Don't confuse risk-neutral and physical parameters. They have different interpretations and uses. Risk-neutral parameters are for pricing; physical parameters are for forecasting and risk measurement.

In [ ]:
def bs_call(S, K, T, r, sigma):
    """Standard Black-Scholes European call price."""
    if T <= 0:
        return max(S - K, 0.0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)

def bs_put(S, K, T, r, sigma):
    """Standard Black-Scholes European put price."""
    call = bs_call(S, K, T, r, sigma)
    return call - S + K * np.exp(-r * T)

def merton_call(S, K, T, r, sigma, lam, mu_J, sigma_J, n_terms=50):
    """Merton jump-diffusion European call price.
    
    Args:
        S: Spot price
        K: Strike price
        T: Time to maturity
        r: Risk-free rate
        sigma: Diffusion volatility
        lam: Jump intensity
        mu_J: Mean of log-jump
        sigma_J: Std of log-jump
        n_terms: Number of series terms
    
    Returns:
        Call price
    """
    k = np.exp(mu_J + 0.5 * sigma_J**2) - 1  # E[J]
    lam_prime = lam * (1 + k)
    
    price = 0.0
    for n in range(n_terms):
        # Poisson weight
        poisson_weight = np.exp(-lam_prime * T) * (lam_prime * T)**n / np.math.factorial(n)
        
        # Adjusted volatility and rate
        sigma_n = np.sqrt(sigma**2 + n * sigma_J**2 / T)
        r_n = r - lam * k + n * np.log(1 + k) / T
        
        price += poisson_weight * bs_call(S, K, T, r_n, sigma_n)
    
    return price

def merton_put(S, K, T, r, sigma, lam, mu_J, sigma_J, n_terms=50):
    """Merton jump-diffusion European put via put-call parity."""
    call = merton_call(S, K, T, r, sigma, lam, mu_J, sigma_J, n_terms)
    return call - S + K * np.exp(-r * T)

# Example pricing
S0_opt = 100
K_opt = 100
T_opt = 0.5
r_opt = 0.05
sigma_opt = 0.20
lam_opt = 1.0
mu_J_opt = -0.10
sigma_J_opt = 0.15

c_bs = bs_call(S0_opt, K_opt, T_opt, r_opt, sigma_opt)
c_merton = merton_call(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mu_J_opt, sigma_J_opt)
p_bs = bs_put(S0_opt, K_opt, T_opt, r_opt, sigma_opt)
p_merton = merton_put(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mu_J_opt, sigma_J_opt)

print(f"{'':20} {'Black-Scholes':>14} {'Merton JD':>14} {'Difference':>12}")
print('-' * 62)
print(f"{'Call Price':20} {c_bs:>14.4f} {c_merton:>14.4f} {c_merton - c_bs:>12.4f}")
print(f"{'Put Price':20} {p_bs:>14.4f} {p_merton:>14.4f} {p_merton - p_bs:>12.4f}")

---
## 7. Series Convergence

The Poisson weights decay factorially, so 5-8 terms suffice for machine precision.

### Convergence properties

The truncation accuracy of the Merton series depends on:
1. **$\lambda T$** — expected number of jumps in the option's life
2. **The desired precision**

For a 1-year option with $\lambda = 1$ jump per year:
* $n=0$ contributes ~37% of total weight
* $n=1$ contributes ~37%
* $n=2$ contributes ~18%
* $n=3$ contributes ~6%
* $n \geq 4$ contributes <2%

So truncating at $n=10$ achieves accuracy below $10^{-7}$ — far beyond what market precision requires.

### When Merton's formula is most needed

The Merton formula matters most for:

**Out-of-the-money options:** Where jumps make the difference between expiring worthless and being deeply ITM. BSM dramatically underestimates these prices.

**Long-dated options:** Where the cumulative effect of jumps grows. BSM's underestimation grows with maturity.

**Stress scenarios:** When designing risk metrics for tail events, the BSM-Merton gap quantifies the model risk.

**Volatility smile interpretation:** The Merton model naturally produces a volatility smile that BSM cannot. Calibrating Merton to market option prices recovers $\lambda, \mu_J, \sigma_J$ that explain the smile shape.

### Beyond Merton — modern jump models

While Merton's lognormal jump-diffusion is the foundational model, modern practice often uses extensions:

**Kou (2002) — Double-exponential jumps:**
$$\log J = \begin{cases} +X^+ \text{ with prob } p \\ -X^- \text{ with prob } 1-p \end{cases}$$

Where $X^+ \sim \text{Exp}(\eta_1)$, $X^- \sim \text{Exp}(\eta_2)$. The double-exponential structure produces heavier tails than Merton's lognormal — empirically a better fit for many markets.

**Variance Gamma (Madan-Seneta 1990):**
A pure jump model (no diffusion). Time-changes Brownian motion by a Gamma process, producing a tractable framework with closed-form characteristic function.

**CGMY (Carr-Geman-Madan-Yor 2002):**
A general framework where the jump activity can be infinite (infinitely many tiny jumps per unit time). Includes Variance Gamma as a special case.

**Bates SVJ (1996):**
Combines Merton jumps with Heston stochastic volatility. The current "industry standard" for equity options. More parameters but better empirical fit.

### Choosing between jump models

| Model | Parameters | Pros | Cons |
|-------|-----------|------|------|
| Merton | 4 | Closed-form, clean | Limited tail flexibility |
| Kou | 5 | Closed-form, double-exp tails | More complex calibration |
| VG | 3 | Pure jump, simple | No diffusion component |
| CGMY | 4 | Most flexible single-process | Numerical methods only |
| Bates SVJ | 7+ | Best empirical fit | Computationally expensive |

For most practical purposes, Merton remains the workhorse because of its tractability. More complex models are reserved for situations where precision matters more than speed (regulatory capital, exotic option pricing, structured product design).

> **Key Concept:** The jump-diffusion world has many models, but they all share Merton's basic structure: continuous diffusion + discrete jumps. The differences lie in the jump distribution and (for some models) whether volatility is also stochastic. Mastering Merton provides the foundation for understanding all of them.

### The Merton series convergence — practical implementation

A well-implemented Merton pricer:

```python
def merton_call(S, K, T, r, sigma, lam, mu_J, sigma_J, n_max=50):
    kappa = np.exp(mu_J + sigma_J**2/2) - 1
    price = 0.0
    for n in range(n_max):
        # Adjusted parameters for n jumps
        sigma_n = np.sqrt(sigma**2 + n * sigma_J**2 / T)
        r_n = r - lam * kappa + n * (mu_J + sigma_J**2/2) / T
        # Poisson weight
        w_n = np.exp(-lam * T) * (lam * T)**n / factorial(n)
        # BSM call with adjusted parameters
        price += w_n * bs_call(S, K, T, r_n, sigma_n)
    return price
```

This implementation:
* Uses log-arithmetic for numerical stability with large $\lambda T$
* Truncates the series at $n_{\max} = 50$ (sufficient for $\lambda T < 30$)
* Handles edge cases (no jumps, very small/large strikes)

### Cross-checking: comparing to Monte Carlo

A standard sanity check: price the same option via Merton series and Monte Carlo. The two should agree within standard error tolerances:

* Merton series: exact (deterministic)
* Monte Carlo: noisy (standard error ~ $\sigma/\sqrt{N}$)

For 1 million Monte Carlo paths, agreement to 4-5 decimal places is achievable. Persistent disagreement indicates a bug in one or the other.

### How big is "big enough"?

For practical option pricing, the question is: how many terms in the Merton series are enough?

**Rule of thumb:** Truncate at $n_{\max} = \max(20, \lambda T + 5\sqrt{\lambda T})$.

For typical parameters:
* $\lambda T = 1$ (one jump per year, 1-year option): $n_{\max} = 20$ is overkill; $n = 10$ suffices
* $\lambda T = 5$ (5 jumps per year, 1-year option): $n_{\max} = 16$
* $\lambda T = 50$ (very high frequency): $n_{\max} = 86$

The series converges *very* quickly because Poisson probabilities decay rapidly past the mean. Even with $\lambda T = 100$, the contribution from $n > 200$ is below numerical precision.

> **Implementation note:** For very large $\lambda T$ (rare in practice but possible for long-dated options on highly volatile assets), the factorial $n!$ in the Poisson weight overflows in standard floating point. Use logarithmic computation: compute $\log P(N_T = n)$ first, then exponentiate.

In [ ]:
# Convergence of Merton series
max_terms = 30
partial_sums = []

k_comp = np.exp(mu_J_opt + 0.5 * sigma_J_opt**2) - 1
lam_p = lam_opt * (1 + k_comp)

running_sum = 0.0
for n in range(max_terms):
    pw = np.exp(-lam_p * T_opt) * (lam_p * T_opt)**n / np.math.factorial(n)
    sigma_n = np.sqrt(sigma_opt**2 + n * sigma_J_opt**2 / T_opt)
    r_n = r_opt - lam_opt * k_comp + n * np.log(1 + k_comp) / T_opt
    running_sum += pw * bs_call(S0_opt, K_opt, T_opt, r_n, sigma_n)
    partial_sums.append(running_sum)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(max_terms), partial_sums, 'o-', color=PRIMARY, markersize=4)
axes[0].axhline(y=partial_sums[-1], color=SECONDARY, linestyle='--', label=f'Converged: {partial_sums[-1]:.4f}')
axes[0].set_xlabel('Number of Terms')
axes[0].set_ylabel('Call Price')
axes[0].set_title('Merton Series Convergence')
axes[0].legend()

# Error vs number of terms
errors = np.abs(np.array(partial_sums) - partial_sums[-1])
errors[errors == 0] = 1e-16  # avoid log(0)
axes[1].semilogy(range(max_terms), errors, 'o-', color=SECONDARY, markersize=4)
axes[1].set_xlabel('Number of Terms')
axes[1].set_ylabel('Absolute Error')
axes[1].set_title('Convergence Error (Log Scale)')

plt.tight_layout()
plt.show()

---
## 8. The Implied Volatility Smile

### Why jumps produce the smile

In the BS world, implied vol should be constant across strikes. Jumps create a **smile/skew**:

1. **OTM puts (low strikes):** Crashes more likely with jumps, so priced higher.
2. **ATM options:** Jump component less important, implied vol near $\sigma$.
3. **OTM calls (high strikes):** Positive jumps also more likely.

> **Key Concept:** The volatility smile is not a "market anomaly" -- it correctly prices jump risk. It says: "the real world has fatter tails than Black-Scholes assumes."

| Parameter change | Effect on smile |
|-----------------|----------------|
| More negative $\mu_J$ | Steeper skew (left side rises) |
| Higher $\sigma_J$ | More symmetric smile |
| Higher $\lambda$ | Smile more pronounced |

### The volatility smile

One of the most empirically observed phenomena in options markets: implied volatilities (computed from market prices using BSM) form a "smile" pattern across strike prices:

* **At-the-money options** have one IV (often called the "ATM IV")
* **Out-of-the-money puts** trade at *higher* IV (called the "skew")
* **Out-of-the-money calls** trade at higher IV too (sometimes), forming a U-shape (the "smile")

For equity index options post-1987, the pattern is more like a **smirk**: high IV for OTM puts, lower IV for OTM calls.

### Why BSM cannot produce a smile

In the BSM model, every option on the same underlying has the same volatility. If you observe different IVs at different strikes, BSM is *internally inconsistent* — there is no single value of $\sigma$ that prices all options correctly.

This is the central empirical failure of BSM. It motivated decades of research into:
* Local volatility (Dupire 1994)
* Stochastic volatility (Heston 1993)
* Jump-diffusion (Merton 1976)

### How jump-diffusion produces a smile

Jump-diffusion *naturally* generates an implied volatility smile. The mechanism:

1. Jumps disproportionately affect deep OTM options
2. Without jumps, OTM options have very low BSM prices (lognormal tails decay fast)
3. With jumps, OTM options have higher prices (jumps push them ITM)
4. To match these higher prices using BSM, you need a *higher* implied volatility
5. Hence: OTM IVs > ATM IVs → the smile

The shape of the smile reveals jump parameters:
* **Asymmetric smirk** ← negative $\mu_J$ (downward jumps dominate)
* **Higher overall smile** ← higher $\lambda$ or higher $\sigma_J$
* **Steeper near ATM** ← higher $\lambda \cdot \sigma_J^2$ ratio

> **CFA Exam Tip:** The volatility smile / skew is heavily tested in the CFA derivatives readings. The exam expects you to know: (1) BSM cannot explain the smile, (2) several models can — including jump-diffusion, stochastic volatility, and local volatility, (3) the smile reflects market expectations of fat tails, especially crash risk for equities.

### The economics of the volatility smile

Why do markets price tail risk above what BSM predicts? Several explanations:

**1. Risk aversion to crashes:**
Investors are *more averse* to large losses than BSM-derived utility implies. They pay a premium for crash protection (OTM puts), pushing those prices above BSM levels.

**2. Hedging demand:**
Portfolio insurers, pension funds, and risk-averse investors *systematically buy* OTM puts. Persistent buying pressure raises put prices beyond BSM levels.

**3. Liquidity premium:**
Deep OTM options are illiquid. Sellers demand a liquidity premium for providing them, raising prices.

**4. Anti-Black-Scholes pressure:**
Post-1987, market makers learned that BSM under-prices tail risk. They charge higher prices to compensate for the model risk.

**5. Tail event probability:**
Markets price in *non-zero probability of catastrophic events*. BSM (with thin tails) doesn't, creating a systematic divergence.

These explanations are not mutually exclusive — all probably contribute. The relative weights vary across markets and time.

### Term structure of the smile

The smile is not just a function of strike — it also varies with maturity:

* **Short-dated options (< 1 month):** Smile is most pronounced. Jumps have outsized effect on near-term volatility.
* **Medium-dated options (1-12 months):** Smile is moderate. Continuous diffusion has more time to dominate.
* **Long-dated options (> 1 year):** Smile flattens. Jump effects "average out" over long horizons.

This term structure is itself information about jump dynamics. Calibrating jump-diffusion across multiple maturities simultaneously gives more reliable parameter estimates than fitting a single maturity.

In [ ]:
def implied_vol_bisection(market_price, S, K, T, r, option_type='call', tol=1e-8, max_iter=200):
    """Find implied volatility using bisection method."""
    low, high = 0.001, 3.0
    price_fn = bs_call if option_type == 'call' else bs_put
    
    for _ in range(max_iter):
        mid = (low + high) / 2
        price = price_fn(S, K, T, r, mid)
        if abs(price - market_price) < tol:
            return mid
        if price > market_price:
            high = mid
        else:
            low = mid
    return mid

# Compute Merton prices across strikes, then back out implied vol
strikes = np.linspace(70, 130, 50)
iv_smile = []

for K in strikes:
    # Merton call price
    c_merton_k = merton_call(S0_opt, K, T_opt, r_opt, sigma_opt, lam_opt, mu_J_opt, sigma_J_opt)
    # Back out implied vol
    iv = implied_vol_bisection(c_merton_k, S0_opt, K, T_opt, r_opt)
    iv_smile.append(iv)

# Different jump parameters for comparison
params_list = [
    (1.0, -0.10, 0.15, 'Negative jumps ($\\mu_J=-0.10$)'),
    (1.0,  0.00, 0.15, 'Symmetric jumps ($\\mu_J=0$)'),
    (1.0,  0.05, 0.15, 'Positive jumps ($\\mu_J=0.05$)'),
    (3.0, -0.05, 0.10, 'Frequent small crashes'),
]

fig, ax = plt.subplots(figsize=(10, 6))
colors = [PRIMARY, SECONDARY, TERTIARY, ACCENT]

for (l, mj, sj, label), color in zip(params_list, colors):
    ivs = []
    for K in strikes:
        c = merton_call(S0_opt, K, T_opt, r_opt, sigma_opt, l, mj, sj)
        ivs.append(implied_vol_bisection(c, S0_opt, K, T_opt, r_opt))
    ax.plot(strikes / S0_opt, np.array(ivs) * 100, label=label, color=color, linewidth=2)

ax.axhline(y=sigma_opt * 100, color='black', linestyle='--', alpha=0.5, label='BSM flat vol')
ax.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Moneyness (K/S)')
ax.set_ylabel('Implied Volatility (%)')
ax.set_title('Implied Volatility Smile from Merton Jump-Diffusion')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 9. Monte Carlo Pricing Under Jump-Diffusion

For path-dependent options where Merton's formula doesn't apply:

$$C \approx e^{-rT} \frac{1}{M}\sum_{m=1}^{M} \text{payoff}(S_T^{(m)})$$

> **Key Concept:** Monte Carlo converges at $O(1/\sqrt{M})$ -- to halve the error, you need 4x more paths. Slow but universal.

### Monte Carlo for jump-diffusion options

When closed-form solutions don't exist (e.g., American options, exotic options under jump-diffusion), Monte Carlo simulation is the workhorse method.

The algorithm:
1. Discretise time into $n$ steps
2. For each path:
   a. At each step, apply the diffusion increment (normal random)
   b. Determine if a jump occurred (Poisson event)
   c. If jump: apply random jump multiplier
3. Compute payoff at expiry
4. Average across many paths
5. Discount to present value

### Variance reduction techniques

Standard Monte Carlo has slow convergence: standard error decreases as $1/\sqrt{n}$. Several techniques accelerate:

* **Antithetic variates:** For each random path, use both the random increment $z$ and its negative $-z$. This halves the variance for many problems.

* **Control variates:** Use a known closed-form (e.g., BSM) as a baseline. Reduce variance by exploiting the correlation between simulated and analytical results.

* **Importance sampling:** Sample more paths in the regions that matter most (e.g., near the strike for ATM options).

* **Stratified sampling:** Divide the random space into strata and sample uniformly from each.

* **Quasi-Monte Carlo:** Use low-discrepancy sequences (Sobol, Halton) instead of pseudo-random numbers. Achieves $O(\log^d n / n)$ convergence vs $O(1/\sqrt{n})$ for standard MC.

These techniques can speed up convergence by orders of magnitude for typical problems.

### Computing Greeks via Monte Carlo

Greeks are sensitivities — partial derivatives of price with respect to inputs. Three approaches:

1. **Finite differences:** Bump input, recompute price, take ratio. Simple but noisy and biased.

2. **Pathwise method:** Differentiate the simulated path with respect to inputs. More accurate, requires problem-specific derivations.

3. **Likelihood ratio method:** Differentiate the *probability density* rather than the path. Works when payoff is non-smooth.

> **Key Concept:** Monte Carlo is the "Swiss army knife" of derivatives pricing — it works for any payoff and any model, given enough computational time. The cost is convergence speed: 100,000 paths is typical for production pricing, 1 million for Greeks, 10 million for tail-risk estimates.

### Path-dependent options under jump-diffusion

Jump-diffusion's flexibility extends naturally to path-dependent options:

**Asian options** (payoff depends on average price):
$$\text{Payoff} = \max\left(\frac{1}{T}\int_0^T S_t \, dt - K, 0\right)$$

Asian options are *less affected* by jumps than European options because averaging dampens the impact of any single discontinuity. BSM Asian prices are reasonably accurate; jump-diffusion adjustments are smaller.

**Barrier options** (payoff contingent on hitting a level):
$$\text{Payoff} = \begin{cases} \max(S_T - K, 0) & \text{if barrier not crossed} \\ 0 & \text{otherwise} \end{cases}$$

Barrier options are *more affected* by jumps. A jump can cause the underlying to leap over a barrier without "touching" it (if observed only at discrete times). The classical BSM barrier formulas assume continuous observation; under jump-diffusion, discrete observation changes the price meaningfully.

**Lookback options** (payoff depends on the maximum or minimum):
Similarly affected by jumps because lookbacks depend on the path's extremes, which jumps can create suddenly.

### The Monte Carlo advantage for path-dependent options

For these complex options, Monte Carlo's flexibility shines. Adding jumps to the simulation is trivial — just add Poisson events at random times and apply jump magnitudes. The same algorithm handles any payoff structure, any underlying dynamics, any path-dependence.

The trade-off: closed-form formulas are usually unavailable, so Monte Carlo is the only option. The associated computational cost (millions of paths for accuracy) is the price of generality.

In [ ]:
def mc_european_jump_diffusion(S0, K, T, r, sigma, lam, mu_J, sigma_J, n_paths, rng, option_type='call'):
    """Monte Carlo pricing of European options under jump-diffusion.
    
    Uses exact terminal distribution (single step).
    """
    k = np.exp(mu_J + 0.5 * sigma_J**2) - 1
    
    # Number of jumps for each path
    n_jumps = rng.poisson(lam * T, size=n_paths)
    
    # Diffusion component
    Z = rng.standard_normal(n_paths)
    diffusion = (r - lam * k - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z
    
    # Jump component
    jump_component = np.zeros(n_paths)
    for i in range(n_paths):
        if n_jumps[i] > 0:
            jump_component[i] = np.sum(rng.normal(mu_J, sigma_J, size=n_jumps[i]))
    
    S_T = S0 * np.exp(diffusion + jump_component)
    
    if option_type == 'call':
        payoffs = np.maximum(S_T - K, 0)
    else:
        payoffs = np.maximum(K - S_T, 0)
    
    price = np.exp(-r * T) * np.mean(payoffs)
    se = np.exp(-r * T) * np.std(payoffs) / np.sqrt(n_paths)
    return price, se

# Compare MC with analytical Merton formula
path_counts = [1000, 5000, 10000, 50000, 100000, 500000]
mc_prices = []
mc_errors = []
analytical = merton_call(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mu_J_opt, sigma_J_opt)

for n in path_counts:
    p, se = mc_european_jump_diffusion(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mu_J_opt, sigma_J_opt, n, rng)
    mc_prices.append(p)
    mc_errors.append(se)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].errorbar(path_counts, mc_prices, yerr=[1.96*se for se in mc_errors], 
                 fmt='o-', color=PRIMARY, capsize=4, label='MC estimate ± 95% CI')
axes[0].axhline(y=analytical, color=SECONDARY, linestyle='--', label=f'Analytical: {analytical:.4f}')
axes[0].set_xscale('log')
axes[0].set_xlabel('Number of Paths')
axes[0].set_ylabel('Call Price')
axes[0].set_title('MC Convergence to Analytical Merton Price')
axes[0].legend()

axes[1].loglog(path_counts, mc_errors, 'o-', color=SECONDARY, label='Standard Error')
# O(1/sqrt(N)) reference
ref = mc_errors[0] * np.sqrt(path_counts[0]) / np.sqrt(path_counts)
axes[1].loglog(path_counts, ref, '--', color='gray', label='$O(1/\\sqrt{N})$')
axes[1].set_xlabel('Number of Paths')
axes[1].set_ylabel('Standard Error')
axes[1].set_title('MC Standard Error Convergence')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 10. Sensitivity to Jump Parameters

| Parameter | Increase causes... |
|-----------|-------------------|
| $\lambda$ | More frequent jumps -- both calls and puts increase |
| $\mu_J$ | More negative: puts rise (crash risk), calls may fall |
| $\sigma_J$ | Fatter tails -- both increase |

> **Key Concept:** Jump risk is fundamentally **non-hedgeable** with the underlying alone. Unlike diffusion risk (delta-hedged), jumps are sudden. The market charges a "jump risk premium."

### Comparing Monte Carlo to closed-form

For European options under Merton's model, both methods should agree:
* **Closed-form (Merton series):** Exact (up to truncation error)
* **Monte Carlo:** Approximate (standard error $\sim 1/\sqrt{n}$)

Their agreement validates both implementations. Disagreement indicates a bug somewhere.

### When Monte Carlo wins

Monte Carlo is preferred when:
1. **No closed-form exists:** American options, exotic options (Asian, lookback, barrier, basket)
2. **Complex underlying dynamics:** Stochastic volatility with jumps, multi-factor models
3. **Path-dependent payoffs:** Cliquet options, ratchet options, autocallables

In these cases, the only alternative — finite-difference PDE methods — can be more complex to implement than Monte Carlo.

### When closed-form wins

Closed-form is preferred when:
1. **Standard European payoffs:** Calls, puts, binary options
2. **Real-time pricing:** Trading desks need millisecond response times
3. **Risk management:** Computing Greeks across thousands of positions
4. **Calibration:** Fitting model parameters to observed market prices

For European Merton options, closed-form is much faster than Monte Carlo and more accurate.

> **CFA Exam Tip:** The CFA exam tests when to use which method. The general principle: if a closed-form exists for the specific option, use it. If not (American, exotic), use Monte Carlo or finite-difference. Monte Carlo is rarely the *fastest* method but is the most *flexible*.

### Jump-diffusion in production trading systems

Real-world trading desks use jump-diffusion models for:

**Real-time pricing:**
Closed-form Merton formulas allow microsecond pricing of European options. Greeks (delta, gamma, vega, theta, jump-vega, jump-rho) are computed similarly.

**Scenario analysis:**
"What's my P&L if jumps double in frequency?" or "What if jump magnitudes increase by 50%?" These scenarios are easily run with calibrated models.

**Hedging:**
Standard delta hedging is supplemented by:
* Vega hedging (against changes in $\sigma$)
* Vomma (volatility-of-volatility) hedging
* Jump risk hedging via short OTM options
* "Synthetic insurance" via dynamic strategies

**Risk attribution:**
Decomposing portfolio P&L into:
* Diffusion contribution
* Jump contribution
* Vega contribution
* Higher-order Greeks

This decomposition tells managers *why* their P&L moved, not just by how much.

**Regulatory capital:**
Basel III's Internal Model Approach (IMA) for market risk requires sophisticated tail-risk models. Many institutions use jump-diffusion or its extensions to compute regulatory capital.

### The CFA exam connection

While the CFA curriculum doesn't require detailed jump-diffusion calculations, the conceptual understanding is essential for advanced derivatives questions:

* "Why does BSM produce systematic pricing errors for OTM options?" → Fat tails / jumps
* "What model would you recommend for crash insurance?" → Jump-diffusion or stochastic vol with jumps
* "How would you hedge a deep OTM put?" → Static option hedge, not just delta hedging

> **Key Concept:** Jump-diffusion models are the *first step* beyond BSM in modelling sophistication. They capture qualitative phenomena (fat tails, smile) that BSM cannot, while remaining tractable enough for daily use. Mastering them is the gateway to more advanced derivatives pricing — stochastic volatility, multi-factor models, exotic options.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sensitivity to lambda (jump intensity)
lam_range = np.linspace(0, 5, 30)
calls_lam = [merton_call(S0_opt, K_opt, T_opt, r_opt, sigma_opt, l, mu_J_opt, sigma_J_opt) for l in lam_range]
puts_lam = [merton_put(S0_opt, K_opt, T_opt, r_opt, sigma_opt, l, mu_J_opt, sigma_J_opt) for l in lam_range]

axes[0].plot(lam_range, calls_lam, color=PRIMARY, linewidth=2, label='Call')
axes[0].plot(lam_range, puts_lam, color=SECONDARY, linewidth=2, label='Put')
axes[0].axhline(y=c_bs, color=PRIMARY, linestyle='--', alpha=0.4, label='BS Call')
axes[0].axhline(y=p_bs, color=SECONDARY, linestyle='--', alpha=0.4, label='BS Put')
axes[0].set_xlabel('Jump Intensity $\\lambda$')
axes[0].set_ylabel('Option Price')
axes[0].set_title('Sensitivity to $\\lambda$')
axes[0].legend(fontsize=9)

# Sensitivity to mu_J (mean jump size)
mj_range = np.linspace(-0.30, 0.10, 30)
calls_mj = [merton_call(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mj, sigma_J_opt) for mj in mj_range]
puts_mj = [merton_put(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mj, sigma_J_opt) for mj in mj_range]

axes[1].plot(mj_range, calls_mj, color=PRIMARY, linewidth=2, label='Call')
axes[1].plot(mj_range, puts_mj, color=SECONDARY, linewidth=2, label='Put')
axes[1].set_xlabel('Mean Log-Jump $\\mu_J$')
axes[1].set_ylabel('Option Price')
axes[1].set_title('Sensitivity to $\\mu_J$')
axes[1].legend(fontsize=9)

# Sensitivity to sigma_J (jump volatility)
sj_range = np.linspace(0.01, 0.40, 30)
calls_sj = [merton_call(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mu_J_opt, sj) for sj in sj_range]
puts_sj = [merton_put(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mu_J_opt, sj) for sj in sj_range]

axes[2].plot(sj_range, calls_sj, color=PRIMARY, linewidth=2, label='Call')
axes[2].plot(sj_range, puts_sj, color=SECONDARY, linewidth=2, label='Put')
axes[2].set_xlabel('Jump Volatility $\\sigma_J$')
axes[2].set_ylabel('Option Price')
axes[2].set_title('Sensitivity to $\\sigma_J$')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 11. Calibration to Market Data

Minimize pricing error across strikes: $\min_{\sigma, \lambda, \mu_J, \sigma_J} \sum_i (C_i^{\text{model}} - C_i^{\text{market}})^2$

### Visualisation insights

The three-panel visualisation reveals key properties of jump-diffusion:

**Panel 1 — Sample paths:**
Smooth diffusion punctuated by occasional sharp drops. The visual contrast with pure GBM paths (uniformly smooth) is striking. Crashes appear as nearly-vertical drops; recoveries are gradual.

**Panel 2 — Terminal distribution:**
Compared to lognormal (BSM), the jump-diffusion terminal distribution shows:
* Heavier left tail (more crashes)
* Negative skew
* Higher kurtosis
* Possible bimodality if $\mu_J$ is large negative

**Panel 3 — Volatility smile:**
Implied volatility plotted against strike (or moneyness). The characteristic smile shape — flat with BSM, smiling with jump-diffusion — is the smoking gun for jump-driven dynamics.

### Reading the smile

The shape of the smile encodes information about the jump parameters:

| Smile Feature | What It Reveals |
|---------------|-----------------|
| Steeper left side (higher OTM put IV) | More negative $\mu_J$ — downward jumps dominate |
| Higher overall level | Higher $\lambda$ or $\sigma_J$ — more jump risk |
| Steeper U-shape | Larger $\sigma_J$ — wider jump distribution |
| Time decay of smile | Faster smile flattening with maturity |

This is **inverse problem**: from the observed smile, infer the model parameters. Active research area; many proposed methods.

### Reading the visualisations carefully

The three visualisation panels each communicate different aspects of jump-diffusion:

**Panel 1 — Sample paths analysis:**
Look for:
* The *frequency* of visible jumps — should match $\lambda T$ on average
* The *direction* of jumps — predominantly down for negative $\mu_J$
* The *magnitude* — controlled by $\sigma_J$
* The *correlation* between consecutive paths (should be zero — paths are independent)

**Panel 2 — Distribution analysis:**
Compare to lognormal:
* Where does jump-diffusion diverge from BSM?
* Are tails fatter on both sides or asymmetric?
* Is there a visible bimodality?
* What's the relative shift of the mean vs median?

**Panel 3 — Volatility smile:**
Identify:
* The IV at the money (ATM IV) — typically the lowest point on the smile
* The slope of the smile — steeper indicates more jump risk
* The asymmetry — usually steeper on the put (left) side for equities
* The level — overall higher means more total volatility (diffusive + jump)

### Quantitative diagnostics

Beyond visual inspection, key statistics include:

* **Higher moments of returns:** Skewness, kurtosis, and beyond
* **Tail probabilities:** $P(R < -5\%)$ for daily returns
* **Maximum drawdown:** The worst peak-to-trough decline
* **Recovery time:** How long until the asset returns to a previous high
* **Jump statistics:** Frequency, average magnitude, and distribution

These statistics are routinely computed and tracked by risk management departments.

In [ ]:
# Generate synthetic market prices from known parameters
true_params = {'sigma': 0.20, 'lam': 1.5, 'mu_J': -0.08, 'sigma_J': 0.12}
calib_strikes = np.array([80, 85, 90, 95, 100, 105, 110, 115, 120])
calib_T = 0.5
calib_r = 0.05

market_prices = np.array([
    merton_call(S0_opt, K, calib_T, calib_r, true_params['sigma'],
                true_params['lam'], true_params['mu_J'], true_params['sigma_J'])
    for K in calib_strikes
])

# Add small noise to simulate real market data
market_prices += rng.normal(0, 0.05, len(market_prices))

def calibration_objective(params):
    """Sum of squared pricing errors."""
    sigma, lam, mu_J, sigma_J = params
    if sigma <= 0 or lam < 0 or sigma_J <= 0:
        return 1e10
    model_prices = np.array([
        merton_call(S0_opt, K, calib_T, calib_r, sigma, lam, mu_J, sigma_J)
        for K in calib_strikes
    ])
    return np.sum((model_prices - market_prices)**2)

# Calibrate using Nelder-Mead
x0 = [0.25, 1.0, -0.05, 0.10]  # initial guess
result = optimize.minimize(calibration_objective, x0, method='Nelder-Mead',
                           options={'xatol': 1e-8, 'fatol': 1e-10, 'maxiter': 5000})

sigma_cal, lam_cal, mu_J_cal, sigma_J_cal = result.x

print(f"{'Parameter':<15} {'True':>10} {'Calibrated':>12}")
print('-' * 40)
for name, true_val, cal_val in [
    ('sigma', true_params['sigma'], sigma_cal),
    ('lambda', true_params['lam'], lam_cal),
    ('mu_J', true_params['mu_J'], mu_J_cal),
    ('sigma_J', true_params['sigma_J'], sigma_J_cal),
]:
    print(f"{name:<15} {true_val:>10.4f} {cal_val:>12.4f}")

# Plot fit
fitted_prices = [merton_call(S0_opt, K, calib_T, calib_r, sigma_cal, lam_cal, mu_J_cal, sigma_J_cal)
                 for K in calib_strikes]

plt.figure(figsize=(10, 6))
plt.plot(calib_strikes, market_prices, 'o', color=SECONDARY, markersize=8, label='Market prices')
plt.plot(calib_strikes, fitted_prices, 's-', color=PRIMARY, markersize=6, label='Calibrated model')
plt.xlabel('Strike Price')
plt.ylabel('Call Price')
plt.title('Merton Jump-Diffusion Calibration')
plt.legend()
plt.tight_layout()
plt.show()

---
## 12. Summary & Extensions

| Concept | Key Result |
|---------|------------|
| Poisson process | Models rare, discrete events with intensity $\lambda$ |
| Merton JD | $dS/S = (\mu - \lambda k)dt + \sigma dW + J dN$ |
| Return distribution | Poisson mixture of normals -- heavy tails |
| Option pricing | Series of BS prices weighted by Poisson PMF |
| Implied volatility | Naturally produces smile/skew |
| Jump risk | Non-hedgeable with underlying alone |

### Calibration — the practical challenge

Calibrating a jump-diffusion model means: given observed market option prices (or implied volatilities), find the model parameters $(\sigma, \lambda, \mu_J, \sigma_J)$ that best reproduce those prices.

### The calibration loss function

Several common choices:

**Sum of squared price errors:**
$$L_1 = \sum_i (C^{model}_i - C^{market}_i)^2$$

**Sum of squared IV errors:**
$$L_2 = \sum_i (IV^{model}_i - IV^{market}_i)^2$$

**Weighted by liquidity:**
$$L_3 = \sum_i w_i (C^{model}_i - C^{market}_i)^2$$

where $w_i$ reflects market liquidity (volume, open interest).

The choice of loss function matters: $L_1$ may overweight ATM options (where prices are higher); $L_2$ weights all strikes more equally.

### Practical calibration

Calibration in practice:
1. Collect option prices across strikes and maturities at a single point in time
2. Choose initial parameter guesses (often previous calibration's results)
3. Run an optimisation (typically Levenberg-Marquardt or similar)
4. Verify the fit by examining residuals
5. Stress-test stability: recalibrate after small data perturbations

A "good" calibration:
* Reproduces market prices to within bid-ask spread
* Has stable parameters across consecutive calibration dates
* Doesn't have wildly different parameters across maturities (model misspecification signal)

> **Common Mistake:** Calibrated parameters are not the same as "true" parameters. They are *implied* parameters — the values that make the model consistent with observed prices. They reflect both the model dynamics *and* market sentiment / risk premia. Don't interpret calibrated $\lambda$ as the actual frequency of crashes — it's the risk-neutral implied frequency that may include risk premium.

### Calibration challenges and pitfalls

Jump-diffusion calibration has several pitfalls beginners often encounter:

**1. Local minima:**
The optimisation landscape can have multiple local minima. Different starting points may converge to different parameters. Solutions: try multiple starting points, use global optimisers (basin hopping, differential evolution), regularise by adding penalties for unreasonable parameters.

**2. Identification problems:**
Different parameter combinations can produce nearly identical option prices. This is especially true between "high frequency, small jumps" and "low frequency, large jumps". The data may not distinguish them well.

**3. Noisy market data:**
Bid-ask spreads, illiquid quotes, and timing differences create noise in the input prices. Calibration should be robust to these — perhaps using only mid prices, weighting by volume, or excluding deep OTM options.

**4. Stability across time:**
A "good" calibration produces parameters that are stable across consecutive trading days. Wild fluctuations indicate model misspecification or data problems.

**5. Forecasting validity:**
Calibrated parameters describe market expectations *at calibration time*. They're not necessarily good forecasts of future jump frequency or magnitude. Don't extrapolate calibrated $\lambda$ as a literal forecast of crash frequency.

### Calibration best practices

Industry best practices:
1. **Use a hierarchical approach:** Calibrate vol surface first, then jumps to fit the residuals
2. **Multiple maturities simultaneously:** Forces consistency across the term structure
3. **Penalty terms:** Prevent parameters from drifting too far from previous values
4. **Robust optimisation:** Use methods that handle non-convex landscapes
5. **Cross-validation:** Hold out some options, calibrate to the rest, test fit on held-out

> **CFA Exam Tip:** Calibration is more an art than a science. The CFA exam may ask conceptual questions about calibration challenges, but the practical work is left to specialised quant teams. Understanding *why* calibration is hard is more important than mastering specific algorithms.

## 13. References

1. Merton, R. C. "Option Pricing When Underlying Stock Returns Are Discontinuous," *JFE*, 1976.
2. Black, F. & Scholes, M. "The Pricing of Options and Corporate Liabilities," *JPE*, 1973.
3. Bates, D. "Jumps and Stochastic Volatility," *RFS*, 1996.
4. Cont, R. & Tankov, P. *Financial Modelling with Jump Processes*, CRC Press, 2003.
5. Hull, J. C. *Options, Futures, and Other Derivatives*, 11th ed., Pearson, 2022.
6. Glasserman, P. *Monte Carlo Methods in Financial Engineering*, Springer, 2003.## 13. References

### Foundational papers

* **Merton, R. C. (1976)** — *"Option pricing when underlying stock returns are discontinuous,"* Journal of Financial Economics, 3(1-2), 125-144. The original jump-diffusion paper with the closed-form formula.

* **Kou, S. G. (2002)** — *"A jump-diffusion model for option pricing,"* Management Science, 48(8), 1086-1101. Introduces the double-exponential jump distribution.

* **Bates, D. S. (1996)** — *"Jumps and stochastic volatility: exchange rate processes implicit in Deutsche Mark options,"* The Review of Financial Studies, 9(1), 69-107. Combines jumps with stochastic volatility (the SVJ model).

### Key extensions

* **Carr, P., Geman, H., Madan, D., & Yor, M. (2002)** — *"The fine structure of asset returns: An empirical investigation,"* Journal of Business. Introduces the CGMY model with infinite jump activity.

* **Duffie, D., Pan, J., & Singleton, K. (2000)** — *"Transform analysis and asset pricing for affine jump-diffusions,"* Econometrica. General framework for affine jump-diffusion pricing.

### Standard textbooks

* **Cont, R., & Tankov, P.** — *Financial Modelling with Jump Processes* (2003). The definitive reference on jump models in finance.

* **Schoutens, W.** — *Lévy Processes in Finance* (2003). Broader coverage of jump models including Variance Gamma, NIG, CGMY.

* **Hull, J. C.** — *Options, Futures, and Other Derivatives*. Brief but accessible coverage of Merton's model.

### CFA curriculum

The CFA curriculum touches jump-diffusion concepts in Level 2 and Level 3 derivatives readings. Key testable points:
* Why BSM fails empirically (fat tails, volatility smile)
* What jump-diffusion adds to BSM
* Trade-offs between jump-diffusion and other extensions (stochastic volatility, local volatility)
* Implications for risk management of tail events

### Key concepts to remember

If you take only one set of ideas from this notebook:

1. **Real markets jump** — empirical fat tails are not statistical artifacts but reflect genuine discontinuous price moves.

2. **BSM is incomplete** — the BSM assumption of continuous paths fails to capture market crashes, news jumps, and earnings surprises.

3. **Jump-diffusion is the canonical fix** — adds a Poisson process to BSM's geometric Brownian motion, producing fat tails and volatility smile.

4. **Closed-form pricing exists** — Merton's series formula prices European options under jump-diffusion. This makes the model practical for production use.

5. **Markets are incomplete under jumps** — jump risk cannot be fully hedged via dynamic delta hedging. Static option hedges and volatility products are needed for complete protection.

6. **Calibration produces risk-neutral parameters** — implied jump frequencies and magnitudes reflect both physical jump dynamics and risk premia.

### Connections to other CFA topics

Jump-diffusion concepts connect to:
* **Equity valuation:** Crash risk premium in expected returns
* **Portfolio management:** Tail-risk allocation strategies
* **Risk management:** Stress testing, VaR, expected shortfall
* **Derivatives:** Volatility smile, smile-fitting models
* **Behavioural finance:** Loss aversion and crash insurance demand

A complete understanding of finance requires fluency with jump dynamics — even at the conceptual level. Jump-diffusion is the rigorous framework that makes such understanding possible.

> **Final thought:** Models are simplifications of reality. BSM simplified equity dynamics to continuous diffusion. Merton's jump-diffusion added a layer of realism by including jumps. Even more sophisticated models (stochastic volatility with jumps, time-changed Lévy processes) layer on additional realism. The progression of model complexity reflects the progression of empirical knowledge about how markets actually behave.